# BIL485 Derin Öğrenme, Ödev 2: Computer Vision
## Food-101 ile yemek fotoğrafı sınıflandırma (sıfırdan CNN ve ResNet-50 transfer learning)

**Problem:** Bir yemek fotoğrafına bakıp onu 101 yemek sınıfından birine atamak (görüntü sınıflandırma).

**Neden bu problem:** Sonuç, Food-101 dışından gerçek yemek fotoğraflarında doğrudan denenebiliyor; Food-101'in 101 sınıfı dengeli ve
birbirine benzeyen yemekler (biftek ve fileto, çikolatalı kek ve çikolatalı mus) modelin nerede zorlandığını görmeyi kolaylaştırıyor.

**Akış**
1. Veri seti ve train / validation / test bölmesi
2. Beklentiler (sonuçları görmeden önce yazıldı)
3. Ortak eğitim düzeni ve hiperparametreler
4. Model 1: sıfırdan eğitilen küçük CNN (temel çizgi)
5. Model 2: ImageNet ile ön-eğitilmiş ResNet-50, ince ayar (transfer learning)
6. Validation'a göre model seçimi, ardından test kümesinde **tek** değerlendirme
7. Hata analizi: sınıf bazında F1, en çok karışan sınıf çiftleri, yanlış örnekler
8. Çıkarım süresi (GPU ve CPU)
9. Food-101 dışından 5 gerçek dünya fotoğrafı + Grad-CAM
10. Uygulama: Gradio web arayüzü
11. Rapor tablosu (`REPORT.md`) ve ham sonuçlar (`results/results.json`)
12. Değerlendirme

**Çalıştırma:** Colab → *Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU* (A100 varsa veri yükleme daha çok CPU
çekirdeğiyle geldiği için belirgin hızlı). Ardından *Tümünü çalıştır*. T4'te toplam süre yaklaşık 1–1,5 saat
(indirme ~10 dk, sıfırdan CNN ~25 dk, ResNet-50 ~30 dk, test ve analiz ~5 dk).

In [ ]:
# 0. Kurulum: Colab'de repoyu klonlar ve eksik paketleri kurar
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/uuu4/bil485-cv-food101"

if IN_COLAB and not os.path.exists("food_cv.py"):
    if not os.path.exists("bil485-cv-food101"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("bil485-cv-food101")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio", "pillow-heif"], check=True)
print("Çalışma klasörü:", os.getcwd())

In [ ]:
import dataclasses, gc, json, shutil, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

import food_cv as fc
from food_cv import TrainConfig

SEED = 42
QUICK_RUN = False   # True: akışı birkaç dakikada uçtan uca dener (sınıf başına 20 görüntü, 1 epoch). Rapor için False.
USE_DRIVE = False   # True: checkpoint'ler Google Drive'a yazılır; oturum düşerse eğitim baştan yapılmaz.

PROJECT_TITLE = "Food-101 ile Yemek Fotoğrafı Sınıflandırma: Sıfırdan CNN ve ResNet-50 Transfer Learning"
REPOSITORY = REPO_URL
DATA_ROOT = Path("/content/data") if IN_COLAB else Path("data")
RESULTS = Path("results")
MY_PHOTOS = Path("my_photos")
CKPT_DIR = Path("checkpoints")
VAL_PER_CLASS = 75                      # resmî train'den sınıf başına validation örneği (750 -> 675 train + 75 val)
NUM_WORKERS = min(8, os.cpu_count() or 2)

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = Path("/content/drive/MyDrive/bil485-cv-food101/checkpoints")
for d in (RESULTS, CKPT_DIR, MY_PHOTOS):
    d.mkdir(parents=True, exist_ok=True)

fc.seed_everything(SEED)
torch.backends.cudnn.benchmark = True
device = fc.get_device()
env = fc.environment_info()
print(json.dumps(env, indent=2, ensure_ascii=False))
if device.type != "cuda":
    print("UYARI: GPU yok. Colab'de Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU seçin.")

## 1. Veri seti ve bölme

**Food-101** (Bossard ve ark., ECCV 2014): 101 yemek sınıfı, sınıf başına 1000 görüntü, toplam 101 000.
Resmî bölünme sınıf başına 750 train + 250 test görüntüsü. Görüntülerin uzun kenarı en fazla 512 piksel.

Bilinmesi gereken bir özellik: veri setinin yazarları **train** görüntülerini kasıtlı olarak temizlememiş; içinde aşırı renkli ve
bazen yanlış etiketli örnekler var. **Test** görüntüleri ise elle kontrol edilmiş. Yani model gürültülü etiketlerden öğrenip temiz
etiketlerde ölçülüyor.

**Bölme kuralı**
- *Train:* resmî train kümesinden sınıf başına 675 görüntü (68 175).
- *Validation:* resmî train kümesinden sınıf başına 75 görüntü (7 575), tabakalı ve sabit tohumla seçilir. Epoch seçimi ve iki model
  arasındaki seçim yalnızca buna göre yapılır.
- *Test:* resmî test kümesi (25 250). Hiçbir karar için kullanılmaz; tüm seçimler bittikten sonra bir kez ölçülür.

In [ ]:
t0 = time.time()
tr_files, tr_labels, te_files, te_labels, classes = fc.food101_file_lists(DATA_ROOT)
print(f"Food-101 hazır ({(time.time() - t0) / 60:.1f} dk). Sınıf sayısı: {len(classes)}")

train_idx, val_idx = fc.stratified_split(tr_labels, VAL_PER_CLASS, SEED)
test_idx = list(range(len(te_files)))
if QUICK_RUN:
    train_idx = fc.subsample_per_class(train_idx, tr_labels, 20, SEED)
    val_idx = fc.subsample_per_class(val_idx, tr_labels, 5, SEED)
    test_idx = fc.subsample_per_class(test_idx, te_labels, 10, SEED)

assert not set(train_idx) & set(val_idx), "train ve validation çakışıyor"
assert not {tr_files[i] for i in train_idx + val_idx} & {te_files[i] for i in test_idx}, "test sızıntısı"

split = {"train": len(train_idx), "val": len(val_idx), "test": len(test_idx), "val_per_class": VAL_PER_CLASS}
counts = pd.DataFrame({
    "train": pd.Series([tr_labels[i] for i in train_idx]).value_counts(),
    "val": pd.Series([tr_labels[i] for i in val_idx]).value_counts(),
    "test": pd.Series([te_labels[i] for i in test_idx]).value_counts(),
})
print(split)
print("Sınıf başına örnek (min / max):", counts.min().to_dict(), counts.max().to_dict())

In [ ]:
# Train kümesinden rastgele örnekler
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for ax, i in zip(axes.flat, rng.choice(train_idx, 18, replace=False)):
    ax.imshow(fc.load_image(tr_files[i]))
    ax.set_title(fc.pretty(classes[tr_labels[i]]), fontsize=9)
    ax.axis("off")
plt.suptitle("Food-101 train örnekleri")
plt.tight_layout()
plt.savefig(RESULTS / "dataset_samples.png", dpi=110, bbox_inches="tight")
plt.show()

## 2. Beklentiler (sonuçları görmeden önce)

1. **Sıfırdan CNN** 6 epoch'ta Food-101'de düşük kalır (%40–55 civarı test doğruluğu). 68 bin görüntü, 101 benzer sınıfı sıfırdan
   ayırt etmeyi öğrenmek için az; kenar, doku ve renk gibi temel özellikleri de bu veriden öğrenmesi gerekiyor.
2. **ResNet-50 transfer learning** aynı sürede çok daha yüksek çıkar (%82–87). ImageNet'ten gelen gövde temel görsel özellikleri
   zaten biliyor; bizim veri yalnız bu özellikleri yemeklere uyarlıyor.
3. En çok karışan sınıflar görünüşü benzeyen yemekler olur: biftek ile fileto, çikolatalı kek ile çikolatalı mus, farklı çorbalar.
4. Gerçek dünya fotoğraflarında Food-101'de olan yemekler büyük ölçüde doğru bilinir. Türk yemekleri gibi **sınıf dışı** yemeklerde model
   yine bir sınıf söylemek zorunda kalır, ama olasılığı düşük olur (ör. lahmacun → pizza).

## 3. Ortak eğitim düzeni

İki model de aynı bölmede, aynı kayıp fonksiyonu ve aynı öğrenme oranı takvimiyle eğitilir:

- **Kayıp:** cross-entropy, label smoothing 0,1 (gürültülü train etiketlerine karşı modelin aşırı emin olmasını azaltır).
- **Optimizer:** AdamW. ResNet-50'de gövde (ön-eğitilmiş katmanlar) sınıflandırıcı başından 10 kat küçük öğrenme oranıyla güncellenir:
  gövde zaten iyi özellikler biliyor, büyük adımlar bunları bozabilir; yeni eklenen son katman ise sıfırdan öğreniyor.
- **Öğrenme oranı takvimi:** ilk yarım epoch doğrusal ısınma, sonra kosinüsle sıfıra iniş.
- **Veri çoğaltma (yalnız train):** rastgele kırpma-ölçekleme (alanın %35–100'ü), yatay çevirme, hafif parlaklık/kontrast/doygunluk.
  Validation ve test: kenarı büyütüp ortadan kare kırpma.
- **Karma hassasiyet (AMP, fp16)** ile GPU'da hızlı eğitim.
- Her epoch sonunda validation ölçülür; **en yüksek validation doğruluğu** veren epoch'un ağırlıkları saklanır.

In [ ]:
CFG_CNN = TrainConfig(name="smallcnn", arch="smallcnn", img_size=128, epochs=6, batch_size=128, lr=2e-3,
                      backbone_lr_mult=1.0, weight_decay=5e-4, warmup_epochs=0.5, label_smoothing=0.1,
                      pretrained=False, dropout=0.3)
CFG_RESNET = TrainConfig(name="resnet50", arch="resnet50", img_size=224, epochs=6, batch_size=64, lr=1e-3,
                         backbone_lr_mult=0.1, weight_decay=1e-4, warmup_epochs=0.5, label_smoothing=0.1,
                         pretrained=True, dropout=0.2)
if QUICK_RUN:
    CFG_CNN = dataclasses.replace(CFG_CNN, epochs=1)
    CFG_RESNET = dataclasses.replace(CFG_RESNET, epochs=1)

display(pd.DataFrame([dataclasses.asdict(CFG_CNN), dataclasses.asdict(CFG_RESNET)]).set_index("name").T)

In [ ]:
def make_split_loaders(img_size, batch_size):
    tr = fc.ImageListDataset([tr_files[i] for i in train_idx], [tr_labels[i] for i in train_idx], fc.train_transform(img_size))
    va = fc.ImageListDataset([tr_files[i] for i in val_idx], [tr_labels[i] for i in val_idx], fc.eval_transform(img_size))
    return (fc.make_loader(tr, batch_size, True, NUM_WORKERS, drop_last=True),
            fc.make_loader(va, 256, False, NUM_WORKERS))


def run_experiment(cfg):
    # Checkpoint varsa (ör. oturum düştü, Drive açık) yeniden eğitmez, kaydedilen geçmişle devam eder.
    ckpt_path = CKPT_DIR / f"{cfg.name}_food101.pt"
    if ckpt_path.exists() and not QUICK_RUN:
        model, meta = fc.load_checkpoint(ckpt_path, device=device)
        if meta["config"] == cfg:
            print(f"{ckpt_path} bulundu, eğitim atlandı.")
            return model, meta["extra"]
        print("Kayıtlı checkpoint farklı ayarlarla eğitilmiş, yeniden eğitiliyor.")
    fc.seed_everything(SEED)
    model = fc.build_model(cfg.arch, len(classes), pretrained=cfg.pretrained, dropout=cfg.dropout)
    train_loader, val_loader = make_split_loaders(cfg.img_size, cfg.batch_size)
    print(f"{cfg.name}: {fc.count_parameters(model) / 1e6:.1f} M parametre, epoch başına {len(train_loader)} adım")
    history, best_epoch, train_time = fc.fit(model, train_loader, val_loader, cfg, device)
    del train_loader, val_loader
    gc.collect()
    extra = {"history": history, "best_epoch": best_epoch, "train_time_s": train_time,
             "best_val_acc": max(h["val_acc"] for h in history)}
    fc.save_checkpoint(model, cfg, classes, ckpt_path, extra=extra)
    return model, extra


def plot_history(history, title, path):
    h = pd.DataFrame(history)
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4))
    a.plot(h.epoch, h.train_loss, "o-", label="train (label smoothing'li)")
    a.plot(h.epoch, h.val_loss, "o-", label="validation")
    a.set_title("Kayıp"); a.set_xlabel("epoch"); a.legend()
    b.plot(h.epoch, h.train_acc, "o-", label="train (çoğaltılmış veri)")
    b.plot(h.epoch, h.val_acc, "o-", label="validation")
    b.plot(h.epoch, h.val_top5, "s--", label="validation top-5")
    b.set_title("Doğruluk"); b.set_xlabel("epoch"); b.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(path, dpi=110, bbox_inches="tight")
    plt.show()

## 4. Model 1: sıfırdan eğitilen küçük CNN (temel çizgi)

Dört evrişim bloğu (her blokta iki 3×3 evrişim + Batch Normalization + ReLU, ardından 2×2 max pooling; kanal sayısı 32 → 64 → 128 → 256),
global ortalama havuzlama, dropout ve 101 çıkışlı tam bağlantılı katman. Giriş 128×128 (sıfırdan eğitimde daha hızlı).
Bu model hiçbir ön bilgi olmadan ne kadar ilerlenebildiğini gösteriyor; ResNet-50'nin getirdiği farkı ölçmek için var.

In [ ]:
model_cnn, run_cnn = run_experiment(CFG_CNN)
plot_history(run_cnn["history"], "Sıfırdan CNN (128×128)", RESULTS / "curves_smallcnn.png")

## 5. Model 2: ResNet-50, ImageNet ağırlıklarıyla ince ayar (transfer learning)

ResNet-50 (He ve ark., 2016): 50 katmanlı, artık bağlantılı (residual) bir CNN. ImageNet'te (1,28 milyon görüntü, 1000 sınıf) eğitilmiş
ağırlıklarla başlıyoruz (`IMAGENET1K_V2`). Son 1000 sınıflı katman çıkarılıp yerine dropout + 101 sınıflı yeni bir katman konuyor;
ardından **tüm ağ** Food-101'de eğitiliyor (gövde küçük, yeni katman büyük öğrenme oranıyla). Giriş 224×224.

In [ ]:
model_res, run_res = run_experiment(CFG_RESNET)
plot_history(run_res["history"], "ResNet-50 transfer learning (224×224)", RESULTS / "curves_resnet50.png")

## 6. Model seçimi (yalnız validation) ve test kümesinde tek değerlendirme

Seçim kuralı: en yüksek validation doğruluğu. Seçim bu hücrede sabitlenir; test sonuçları ondan sonra hesaplanır ve hiçbir ayarı
değiştirmek için kullanılmaz. İki modelin test sonucu da raporlanır (karşılaştırma için), ama seçilen model önceden belli.

In [ ]:
runs = {"smallcnn": (CFG_CNN, model_cnn, run_cnn, "Sıfırdan CNN (4 blok)"),
        "resnet50": (CFG_RESNET, model_res, run_res, "ResNet-50 (ImageNet ön-eğitimli, ince ayar)")}
selected = max(runs, key=lambda k: runs[k][2]["best_val_acc"])
print("Validation doğrulukları:", {k: round(v[2]["best_val_acc"], 4) for k, v in runs.items()})
print("Seçilen model:", selected)

test_results, test_outputs = {}, {}
for name, (cfg, model, run, _) in runs.items():
    te_ds = fc.ImageListDataset([te_files[i] for i in test_idx], [te_labels[i] for i in test_idx], fc.eval_transform(cfg.img_size))
    out = fc.evaluate(model, fc.make_loader(te_ds, 256, False, NUM_WORKERS), device, return_outputs=True)
    test_outputs[name] = out
    test_results[name] = fc.classification_metrics(out["logits"], out["labels"], classes)

display(pd.DataFrame({runs[k][3]: {"val acc": runs[k][2]["best_val_acc"], "test acc": m["accuracy"], "test top-5": m["top5_accuracy"],
                                   "macro precision": m["macro_precision"], "macro recall": m["macro_recall"],
                                   "macro F1": m["macro_f1"], "eğitim süresi (dk)": runs[k][2]["train_time_s"] / 60}
                      for k, m in test_results.items()}).round(4))

## 7. Hata analizi (seçilen model)

In [ ]:
m = test_results[selected]
pc = pd.DataFrame(m["per_class"]).sort_values("f1")
pc["class"] = pc["class"].map(fc.pretty)
print("En zor 10 sınıf (F1'e göre):")
display(pc.head(10).round(3).reset_index(drop=True))
print("En kolay 10 sınıf:")
display(pc.tail(10).iloc[::-1].round(3).reset_index(drop=True))
print("En çok karışan sınıf çiftleri (gerçek → tahmin):")
display(pd.DataFrame(m["top_confused_pairs"]).assign(true=lambda d: d.true.map(fc.pretty), pred=lambda d: d.pred.map(fc.pretty),
                                                       rate=lambda d: d.rate.round(3)))

In [ ]:
# En zor 15 sınıf arasındaki karışıklık matrisi (101×101 okunmuyor)
worst = [classes.index(c) for c in pd.DataFrame(m["per_class"]).sort_values("f1")["class"].head(15)]
cm = np.asarray(m["confusion_matrix"])
sub = cm[np.ix_(worst, worst)]
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(sub, cmap="Blues")
ax.set_xticks(range(len(worst)), [fc.pretty(classes[i]) for i in worst], rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(worst)), [fc.pretty(classes[i]) for i in worst], fontsize=8)
for i in range(len(worst)):
    for j in range(len(worst)):
        if sub[i, j]:
            ax.text(j, i, sub[i, j], ha="center", va="center", fontsize=7, color="white" if sub[i, j] > sub.max() / 2 else "black")
ax.set_xlabel("tahmin"); ax.set_ylabel("gerçek")
ax.set_title("En zor 15 sınıf: karışıklık matrisi (test)")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(RESULTS / "confusion_worst15.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# Yanlış sınıflandırılmış test örnekleri
out = test_outputs[selected]
probs = torch.tensor(out["logits"]).softmax(1).numpy()
wrong = np.flatnonzero(out["logits"].argmax(1) != out["labels"])
pick = np.random.default_rng(SEED).choice(wrong, min(12, len(wrong)), replace=False)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, k in zip(axes.flat, pick):
    p = probs[k].argmax()
    ax.imshow(fc.load_image(te_files[test_idx[k]]))
    ax.set_title(f"gerçek: {fc.pretty(classes[out['labels'][k]])}\ntahmin: {fc.pretty(classes[p])} (%{100 * probs[k, p]:.0f})", fontsize=8)
    ax.axis("off")
plt.suptitle("Yanlış sınıflandırılan test örnekleri")
plt.tight_layout()
plt.savefig(RESULTS / "test_errors.png", dpi=110, bbox_inches="tight")
plt.show()

## 8. Çıkarım süresi

Tek görüntü (batch 1) için modelin ileri geçiş süresi, ısınmadan sonra 50 ölçümün medyanı. Ön işleme (okuma, yeniden boyutlandırma)
dahil değil. CPU ölçümü, uygulamanın GPU'suz bir sunucuda çalışacağı durumu gösterir.

In [ ]:
inference = {}
for name, (cfg, model, run, _) in runs.items():
    gpu_ms = fc.time_inference(model, device, cfg.img_size) if device.type == "cuda" else None
    cpu_model = fc.build_model(cfg.arch, len(classes), pretrained=False, dropout=cfg.dropout)
    cpu_model.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()})
    cpu_ms = fc.time_inference(cpu_model, torch.device("cpu"), cfg.img_size, n_warmup=3, n_runs=20)
    inference[name] = {"gpu_ms": gpu_ms, "cpu_ms": cpu_ms, "gpu": env["device"] if gpu_ms else None}
    del cpu_model
display(pd.DataFrame(inference).T)

## 9. Food-101 dışından gerçek dünya fotoğrafları

Bu sürümde `my_photos/` klasöründeki 5 fotoğraf **telefonla çekilmedi, Wikimedia Commons'tan alındı** (açık lisanslı; yazar ve lisanslar `my_photos/SOURCES.md`'de). Hiçbiri Food-101'den değil. Kendi telefon fotoğraflarınızla değiştirmek için dosyaları aynı kurala göre koymanız yeterli. **Dosya adı gerçek etiketi söyler:** Food-101 sınıf adı
(ör. `pizza.jpg`, `french_fries.jpg`, `ice_cream-2.heic`), sınıf dışı bir yemekse kendi adı (ör. `lahmacun.jpg`). Aynı sınıftan ikinci
fotoğraf için `-2` eklenir. Telefonun yan kaydettiği fotoğrafların yönü EXIF bilgisinden düzeltilir; iPhone HEIC dosyaları da okunur.

Her fotoğraf için: modelin gördüğü kare, Grad-CAM ısı haritası ve ilk 5 tahmin.

In [ ]:
if IN_COLAB and not fc.list_images(MY_PHOTOS):
    from google.colab import files
    print("my_photos/ boş: fotoğrafları seçin (dosya adı = gerçek etiket).")
    for fname, data in files.upload().items():
        (MY_PHOTOS / fname).write_bytes(data)
converted = fc.convert_heic_to_jpeg(MY_PHOTOS)
if converted:
    print("HEIC -> JPEG:", [p.name for p in converted])

model_sel, cfg_sel = runs[selected][1], runs[selected][0]
my_rows = fc.evaluate_my_photos(model_sel, MY_PHOTOS, classes, cfg_sel.img_size, device)
if not my_rows:
    print("my_photos/ içinde fotoğraf yok; bu bölüm atlandı.")
else:
    display(pd.DataFrame([{"dosya": r["file"], "gerçek": fc.pretty(r["true_label"]), "Food-101'de": r["in_food101"],
                           "tahmin": fc.pretty(r["pred_top1"]), "olasılık": round(r["pred_top1_prob"], 3),
                           "doğru (top-1)": r["correct_top1"], "doğru (top-5)": r["correct_top5"],
                           "gerçek sınıfın sırası": r["true_rank"]} for r in my_rows]))
    fig, axes = plt.subplots(len(my_rows), 3, figsize=(14, 4.2 * len(my_rows)), squeeze=False)
    for (a0, a1, a2), r in zip(axes, my_rows):
        img = fc.load_image(MY_PHOTOS / r["file"])
        cam, view, _ = fc.grad_cam(model_sel, img, cfg_sel.img_size, device, class_idx=classes.index(r["pred_top1"]))
        a0.imshow(view); a0.set_title(f"{r['file']}\ngerçek: {fc.pretty(r['true_label'])}", fontsize=10); a0.axis("off")
        a1.imshow(fc.overlay_cam(view, cam)); a1.set_title(f"Grad-CAM: {fc.pretty(r['pred_top1'])}", fontsize=10); a1.axis("off")
        names = [fc.pretty(c) for c, _ in r["top5"]][::-1]
        vals = [p for _, p in r["top5"]][::-1]
        colors = ["tab:green" if c == r["true_label"] else "tab:gray" for c, _ in r["top5"]][::-1]
        a2.barh(names, vals, color=colors); a2.set_xlim(0, 1); a2.set_title("İlk 5 tahmin (yeşil = gerçek sınıf)", fontsize=10)
    plt.tight_layout()
    plt.savefig(RESULTS / "my_photos_predictions.png", dpi=110, bbox_inches="tight")
    plt.show()

## 10. Uygulama: Gradio web arayüzü

`app.py`, kaydedilen checkpoint'i yükleyip bir web arayüzü açar: kullanıcı fotoğraf yükler (ya da kamerayla çeker), model ilk 5 tahmini
olasılıklarıyla ve **Grad-CAM** ısı haritasıyla (modelin karar verirken en çok baktığı bölge) gösterir. En olası sınıfın olasılığı %40'ın
altındaysa "fotoğraf 101 sınıftan birine ait olmayabilir" uyarısı verir. §9'daki fotoğraflar arayüzde örnek olarak hazır gelir.

Aşağıdaki hücre `share=True` ile geçici herkese açık bir bağlantı üretir (Colab'de 72 saat geçerli). Bilgisayarda çalıştırmak için:
`python app.py` (checkpoint `checkpoints/` altında olmalı).

In [ ]:
from app import create_demo

demo = create_demo(CKPT_DIR / f"{selected}_food101.pt", device=device, examples_dir=MY_PHOTOS)

# Arayüzün tahmin fonksiyonunu örnek fotoğraflarla sına (arayüze yüklenen fotoğrafla aynı yol)
for p in fc.list_images(MY_PHOTOS):
    _, _, msg = demo.fn(str(p))
    print(f"{p.name}: {msg.splitlines()[0]}")

if IN_COLAB:
    demo.launch(share=True)   # geçici herkese açık bağlantı (72 saat geçerli)
else:
    print("Arayüzü açmak için: python app.py  (ya da --share)")

## 11. Rapor tablosu ve ham sonuçlar

Ödev belgesinin 9. bölümündeki şablon `results/results.json`'dan otomatik doldurulup `REPORT.md`'ye yazılır.
Colab'de sonunda `results/` ve seçilen checkpoint tek zip olarak indirilir (repoya eklemek için).

In [ ]:
results = {
    "project_title": PROJECT_TITLE,
    "repository": REPOSITORY,
    "seed": SEED,
    "quick_run": QUICK_RUN,
    "environment": env,
    "split": split,
    "classes": classes,
    "selected_model": selected,
    "selection_rule": "en yüksek validation accuracy (test kullanılmadı)",
    "models": {
        name: {
            "architecture": label,
            "config": dataclasses.asdict(cfg),
            "params_m": fc.count_parameters(model) / 1e6,
            "history": run["history"],
            "best_epoch": run["best_epoch"],
            "best_val_acc": run["best_val_acc"],
            "train_time_s": run["train_time_s"],
            "test": test_results[name],
            "inference_ms": inference[name],
        }
        for name, (cfg, model, run, label) in runs.items()
    },
    "my_photos": my_rows,
}
fc.save_json(results, RESULTS / "results.json")
fc.write_report(json.loads((RESULTS / "results.json").read_text(encoding="utf-8")), "REPORT.md")
display(Markdown(Path("REPORT.md").read_text(encoding="utf-8")))

In [ ]:
if IN_COLAB:
    bundle = Path("/content/bil485_odev2_sonuclar")
    shutil.rmtree(bundle, ignore_errors=True)
    (bundle / "checkpoints").mkdir(parents=True)
    shutil.copytree(RESULTS, bundle / "results")
    shutil.copy("REPORT.md", bundle / "REPORT.md")
    shutil.copy(CKPT_DIR / f"{selected}_food101.pt", bundle / "checkpoints")
    for p in fc.list_images(MY_PHOTOS):
        (bundle / "my_photos").mkdir(exist_ok=True)
        shutil.copy(p, bundle / "my_photos")
    archive = shutil.make_archive(str(bundle), "zip", bundle)
    from google.colab import files
    files.download(archive)

## 12. Değerlendirme

*(Colab çıktıları geldikten sonra, §2'deki beklentilerle karşılaştırılarak yazılacak.)*

- Genel performans ve iki modelin farkı
- Öğrenme eğrileri: aşırı öğrenme var mı
- En çok karışan sınıflar ve nedeni
- Gerçek dünya fotoğrafları: başarılı ve başarısız olduğu durumlar, Grad-CAM'in gösterdikleri
- Sınırlılıklar